# 中国汽车市场分析：销量预测、产品配置与用户需求

## 数据分析笔记本

本 Notebook 完整记录项目已完成的各阶段工作，展示每一步的实际操作与成果。当前已覆盖：

- **阶段一 · 数据准备**：太平洋月度销量 + 汽车之家/太平洋车型配置两类公开数据，按 `series_name` 直接对齐
- **阶段二 · 数据筛选与探索性可视化**：in-population 子集、销量趋势、车型/级别/能源分布
- **阶段三 · 销量预测建模（腿B）**：ARIMA / Prophet / XGBoost / LSTM / 融合多模型对比、特征消融、预测区间
- **阶段四 · 配置→销量归因（腿A）**：车系×年横截面 XGBoost 归因，量化配置对销量的解释力
- **阶段五 · 舆情融合预测与话题预警（Phase B）**：情感作为外生变量接入销量预测（待接入新数据）
- **阶段六 · 网页看板交付**：Flask + ECharts 7 屏交互式可视化，汇总并呈现前阶段结论

## 1. 环境与路径

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

BASE = os.path.abspath('..')  # notebook 位于项目根目录下一级
RAW = os.path.join(BASE, 'data', 'raw')
SENTIMENT = os.path.join(BASE, 'data', 'sentiment')
PROC = os.path.join(BASE, 'data', 'processed_new')   # new pipeline artifacts
STAGE3 = os.path.join(PROC, 'stage3')
STAGE4 = os.path.join(PROC, 'stage4')
SPLITS = os.path.join(PROC, 'splits')
FIG = os.path.join(BASE, 'figures')

for d in [PROC, STAGE3, STAGE4, FIG]:
    os.makedirs(d, exist_ok=True)

print('Project root:', BASE)
print('Raw data dir :', RAW)
print('New proc dir :', PROC)
print('Sentiment dir:', SENTIMENT)


## 2. 阶段一：数据准备

阶段一是整个项目的数据地基。新管线只依赖两份原始数据——太平洋汽车月度销量与汽车之家/太平洋车型配置，按 `series_name` 直接对齐（无需旧 `series_mapping` 桥接表）。所有原始数据体积较大且不入库（可由 `scripts/new_pipeline/` 完整复现）。

### 2.1 一次性加载两份核心数据集

In [ ]:
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': [
        'PingFang SC', 'Heiti SC', 'Hiragino Sans GB',
        'SimHei', 'Noto Sans CJK SC', 'Microsoft YaHei',
        'DejaVu Sans', 'Arial', 'Helvetica', 'sans-serif'
    ],
    'axes.unicode_minus': False,
    'axes.edgecolor': '#333333',
    'axes.labelcolor': '#333333',
    'text.color': '#333333',
    'xtick.color': '#555555',
    'ytick.color': '#555555',
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'savefig.facecolor': 'white',
    'axes.grid': True,
    'grid.color': '#e0e0e0',
    'grid.linestyle': '-',
    'grid.linewidth': 0.5,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

COLORS = {
    'blue': '#2E86AB', 'orange': '#F18F01', 'green': '#3E8914',
    'red': '#C73E1D', 'purple': '#6A4C93', 'teal': '#1B998B', 'gray': '#8D99AE'
}


### 2.2 月度销量数据（来源：太平洋汽车）

记录每个车系逐月销量，是后续销量预测的目标变量（Y）来源。

### 2.3 车型配置数据（来源：汽车之家/太平洋，粒度=车系×年）

一行一个车系在某一年款，含价格、能源类型、续航、加速等配置字段，是特征变量（X）来源。

In [ ]:
# New pipeline: two raw sources + sentiment (Phase B).
sales = pd.read_csv(os.path.join(RAW, 'monthly_sales.csv'))        # monthly sales (pcauto)
feat = pd.read_csv(os.path.join(RAW, 'feature.csv'))               # configs (series x year)
reviews = pd.read_csv(os.path.join(SENTIMENT, 'sentiment_reviews.csv'))   # raw reviews (Phase B)
senti = pd.read_csv(os.path.join(SENTIMENT, 'sentiment_summary.csv'))     # series-level sentiment (Phase B)

print('monthly_sales :', sales.shape, '| series:', sales.series_name.nunique())
print('feature       :', feat.shape, '| series:', feat.series_name.nunique())
print('reviews (PB)  :', reviews.shape, '| series:', reviews.series_id.nunique())
print('senti (PB)    :', senti.shape)


### 2.4 按 `series_name` 对齐（无需跨平台桥接）

两套数据统一用 `series_name` 关联，得到 371 个「月度+配置」车系（腿B）与 736 个有年度销量车系（腿A）；仅销量无配置的车系排除。

In [ ]:
print('=== Monthly sales - source: PCauto ===')
print('Shape     :', sales.shape)
print('Columns   :', sales.columns.tolist())
print('Date range:', sales.period.min(), '->', sales.period.max(), '(year', sales.year.min(), '~', sales.year.max(), ')')
print('Unique series:', sales.series_name.nunique(), '| Unique brands:', sales.brand.nunique())
print('Total sales  :', int(sales.monthly_sales.sum()), 'units')
print('Zero-share   : %.1f%% (suspension / pre-launch gaps)' % (100*(sales.monthly_sales==0).mean()))
print()
print(sales.head(3).to_string())


### 2.5 舆情数据（Phase B）

来自项目前期懂车帝口碑采集（490 车系，2026-07），属 Phase B 素材；待按 `series_name` 重新对齐到 371/736 新口径后接入。

In [ ]:
print('=== Vehicle configs - source: autohome / pcauto (series x year) ===')
print('Shape:', feat.shape, '(1 row = 1 series in 1 model year, %d columns)' % feat.shape[1])
print('Granularity unique (series_name, year):', feat[['series_name','year']].drop_duplicates().shape[0]==len(feat))
print('Series with annual_sales:', feat.annual_sales.notna().sum(), '/', len(feat))
key = ['series_name','year','brand_name','vehicle_class','energy_type','official_price_wan','annual_sales']
print(feat[key].head(5).to_string())


### 2.6 舆情概览可视化

In [ ]:
# Align the two sources directly by series_name (no legacy ID bridge needed).
ms_series = set(sales.series_name.unique())
feat_series = set(feat.series_name.unique())
inter = ms_series & feat_series
in_pop = len(inter)                       # Leg B in-population series
legA = feat[(feat.annual_sales.notna()) & (feat.year.between(2022, 2026))]['series_name'].nunique()   # Leg A series w/ annual sales in window
sales_only = len(ms_series - feat_series) # excluded (no config)

print('monthly_sales series :', len(ms_series))
print('feature series       :', len(feat_series))
print('intersection (Leg B) :', in_pop)
print('Leg A (annual sales) :', legA)
print('sales-only excluded  :', sales_only)

# Build a small alignment preview
prev = (sales.groupby('series_name')['monthly_sales'].sum()
            .reset_index().rename(columns={'monthly_sales':'total_sales'})
            .merge(feat[['series_name','energy_type','vehicle_class','official_price_wan']]
                   .drop_duplicates('series_name'), on='series_name', how='inner'))
print('\nAligned preview (head):')
print(prev.head(6).to_string())


### 2.7 对齐成果：舆情×销量关系（Phase B）

按 `series_name` 把舆情与销量对齐，画正面评价比例 vs 月均销量散点，直观检验口碑与销量的关联（Phase B 参考）。

In [ ]:
# Phase B: legacy dongchedi sentiment collection (490 series, 2026-07).
print('=== Sentiment (Phase B, source: dongchedi) ===')
print('Total reviews :', len(reviews))
print('Series covered:', reviews.series_id.nunique())
print('Avg length     :', round(reviews.content_len.mean(), 1), 'chars')
sc = senti.positive_cnt.sum(); nc = senti.neutral_cnt.sum(); neg = senti.negative_cnt.sum()
print('Polarity (scored): +%.0f / =%.0f / -%.0f  (pos %.0f%%)' % (
    sc, nc, neg, 100*sc/(sc+nc+neg)))
print('NOTE: Phase B will re-align this to the new 371/736 series by series_name.')


### 2.8 数据质量与覆盖检查

In [ ]:
# Sentiment landscape (Phase B reference, real data).
valid = reviews[reviews.rating_overall > 0]
tot_pos = int(senti.positive_cnt.sum()); tot_neu = int(senti.neutral_cnt.sum()); tot_neg = int(senti.negative_cnt.sum())

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].hist(valid.rating_overall, bins=30, color=COLORS['blue'], edgecolor='white', alpha=0.85)
axes[0].axvline(valid.rating_overall.median(), color=COLORS['orange'], linestyle='--',
                linewidth=2, label='Median: %.1f' % valid.rating_overall.median())
axes[0].set_title('Review Rating Distribution (%d reviews)' % len(valid))
axes[0].set_xlabel('Overall Rating (0-5)'); axes[0].set_ylabel('Number of Reviews')
axes[0].legend(loc='upper left', frameon=False)

labels = ['Positive', 'Neutral', 'Negative']
vals = [tot_pos, tot_neu, tot_neg]; cols = [COLORS['green'], COLORS['gray'], COLORS['red']]
bars = axes[1].bar(labels, vals, color=cols)
axes[1].set_title('Sentiment Composition (all series)'); axes[1].set_ylabel('Number of Reviews')
total = sum(vals)
for b, v in zip(bars, vals):
    axes[1].annotate('%d\n(%.0f%%)' % (v, 100*v/total), xy=(b.get_x()+b.get_width()/2, b.get_height()),
                     xytext=(0, 4), textcoords='offset points', ha='center', va='bottom', fontsize=10, fontweight='bold')
fig.tight_layout(); fig.savefig(os.path.join(FIG, 'sentiment_overview.png'), dpi=150, bbox_inches='tight'); plt.show()


### 2.9 阶段一产出汇总

In [ ]:
# Phase B scatter: positive review ratio vs (log) average monthly sales, aligned by series_name.
try:
    agg = sales.groupby('series_name')['monthly_sales'].mean().reset_index().rename(columns={'monthly_sales':'avg_monthly'})
    df = senti.merge(agg, on='series_name', how='inner').dropna(subset=['positive_ratio','avg_monthly'])
    df = df[df.avg_monthly > 0]
    df['log_avg'] = np.log10(df['avg_monthly'])
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(df['positive_ratio'], df['log_avg'], s=28, alpha=0.6, color=COLORS['blue'])
    z = np.polyfit(df['positive_ratio'], df['log_avg'], 1)
    xs = np.linspace(df['positive_ratio'].min(), df['positive_ratio'].max(), 50)
    ax.plot(xs, np.polyval(z, xs), color=COLORS['orange'], linewidth=2, linestyle='--', label='Linear trend')
    corr = df['positive_ratio'].corr(df['log_avg'])
    ax.set_title('Positive Review Ratio vs Sales (%d series, Phase B, r=%.2f)' % (len(df), corr))
    ax.set_xlabel('Positive Review Ratio'); ax.set_ylabel('log10(Avg Monthly Sales)')
    ax.legend(loc='lower right', frameon=False)
    fig.tight_layout(); fig.savefig(os.path.join(FIG, 'sentiment_vs_sales.png'), dpi=150, bbox_inches='tight'); plt.show()
except Exception as e:
    print('Alignment skipped (Phase B):', e)


## 3. 阶段二：数据筛选与探索性可视化

阶段二完成三件事：

- 确定建模子集（371 个 in-population 车系）。
- 绘制全市场销量趋势、车型分类 / 级别 / 能源类型分布。
- 基于绝对时间切分（train / val / test）准备时序评估。

In [ ]:
print('Monthly-sales duplicated rows :', sales.duplicated().sum())
print('Feature duplicated rows      :', feat.duplicated().sum())
print()
print('Series coverage:')
print('  monthly_sales :', sales.series_name.nunique())
print('  feature       :', feat.series_name.nunique())
print('  sentiment(PB) :', senti.series_name.nunique() if 'series_name' in senti.columns else senti.series_id.nunique())
print()
print('Feature missing-value share (top 5 columns):')
miss = (feat.isna().mean().sort_values(ascending=False) * 100).round(1)
print(miss.head(5).to_string())


### 3.1 时间索引与连续覆盖辅助函数

In [ ]:
stage1_outputs = {
    'monthly_sales.csv (pcauto)':  sales.shape,
    'feature.csv (series x year)': feat.shape,
    'sentiment_reviews.csv (PB)':  reviews.shape,
    'sentiment_summary.csv (PB)':  senti.shape,
}
for k, v in stage1_outputs.items():
    print('%-30s %6d rows x %3d cols' % (k, v[0], v[1]))
print('\nStage 1 complete: two raw sources aligned by series_name; sentiment held for Phase B.')


### 3.2 英文标签映射

图表使用英文标签以避免中文字体缺失导致的乱码；把中文分类值（车型类别 / 级别 / 能源类型）映射为英文。

### 3.3 建模子集（371 in-population 车系）

In [ ]:
sales['date'] = pd.to_datetime(dict(year=sales.year, month=sales.month, day=1))

def runs_info(periods):
    """Return (longest_run, interrupt_count, longest_gap, total_months)."""
    p = np.sort(np.unique(periods))
    if len(p) == 0:
        return 0, 0, 0, 0
    diffs = np.diff(p); runs, gaps, cur = [], [], 1
    for d in diffs:
        if d == 1: cur += 1
        else:
            runs.append(cur); gaps.append(d - 1); cur = 1
    runs.append(cur)
    longest = int(max(runs)); n_interrupt = int(np.sum(diffs > 1))
    longest_gap = int(max(gaps)) if gaps else 0
    return longest, n_interrupt, longest_gap, len(p)


### 3.4 销量趋势可视化

In [ ]:
CATEGORY_MAP = {'SUV': 'SUV', '轿车': 'Sedan', 'MPV': 'MPV'}
VEHICLE_CLASS_MAP = {
    '中型车': 'Mid-size Sedan', '中大型车': 'Large Sedan', '中型SUV': 'Mid-size SUV',
    '紧凑型SUV': 'Compact SUV', '紧凑型车': 'Compact Sedan', '中大型SUV': 'Large SUV',
    '小型SUV': 'Small SUV', '大型SUV': 'Full-size SUV', '中大型MPV': 'Large MPV',
    '小型车': 'Small Sedan', '微型车': 'Mini Car', '中型MPV': 'Mid-size MPV',
    '紧凑型MPV': 'Compact MPV', '大型车': 'Full-size Sedan', '大型MPV': 'Full-size MPV',
    '微面': 'Mini Van', '轻客': 'Light Van', 'MPV': 'MPV',
}
ENERGY_TYPE_MAP = {
    '燃油': 'Gasoline', '纯电动': 'BEV', '插电混动': 'PHEV', '增程式': 'EREV',
    '油电混动': 'HEV', '插混+纯电': 'PHEV+BEV', '其他': 'Other',
}
sales['category_en'] = sales['category'].map(CATEGORY_MAP)
feat['vehicle_class_en'] = feat['vehicle_class'].map(VEHICLE_CLASS_MAP)
feat['energy_type_en'] = feat['energy_type'].map(ENERGY_TYPE_MAP)
print('Mappings applied.')


### 3.5 车型分类 / 级别 / 能源分布可视化

In [ ]:
# New modeling population: 371 in-population series (monthly panel + configs).
manifest = json.load(open(os.path.join(SPLITS, 'manifest.json')))
print('Split n_series :', manifest['n_series'])
print('Time cutoffs   : train_end', manifest['time_cutoffs']['train_end'],
      '| val_end', manifest['time_cutoffs']['val_end'], '| test_end', manifest['time_cutoffs']['test_end'])
print('n_rows         :', manifest['n_rows'])
train = pd.read_csv(os.path.join(SPLITS, 'train.csv'))
print('train rows     :', len(train), '| series:', train.series_name.nunique())


### 3.6 阶段二产出汇总

In [ ]:
# Sales trend of the in-population (371-series) panel.
mt = (sales.groupby('date')['monthly_sales'].sum().reset_index().sort_values('date'))
mt['rolling_12m'] = mt['monthly_sales'].rolling(window=12, min_periods=1).mean()
fig, ax = plt.subplots(figsize=(12, 5.5))
ax.fill_between(mt['date'], mt['monthly_sales'], color=COLORS['blue'], alpha=0.12)
ax.plot(mt['date'], mt['monthly_sales'], color=COLORS['blue'], linewidth=2, label='Monthly sales')
ax.plot(mt['date'], mt['rolling_12m'], color=COLORS['orange'], linewidth=2, label='12-month moving average')
ax.set_title('Monthly Sales Trend (all %d pcauto series)' % sales.series_name.nunique())
ax.set_xlabel('Month'); ax.set_ylabel('Total Sales (units)')
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'))
ax.legend(loc='upper left', frameon=False); ax.set_axisbelow(True)
fig.tight_layout(); fig.savefig(os.path.join(FIG, 'sales_trend.png'), dpi=150, bbox_inches='tight'); plt.show()


## 4. 阶段三：销量预测建模（腿B）

阶段三在 371 个 in-population 车系上构建并对比多类预测模型，按**绝对时间**切分（train 2022-01~2025-06 / val 2025-07~2025-12 / test 2026-01~2026-06），以 150 个车系的分层子集作为统一评估集。

In [ ]:
# Category / vehicle-class / energy-type distributions from the new data.
cat = sales.groupby('category_en')['series_name'].nunique().sort_values(ascending=False)
vclass = feat.drop_duplicates('series_name')['vehicle_class_en'].value_counts().sort_values(ascending=False)
energy = feat.drop_duplicates('series_name')['energy_type_en'].value_counts().sort_values(ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
axes[0].bar(cat.index, cat.values, color=COLORS['blue']); axes[0].set_title('Sales Category (series)'); axes[0].set_ylabel('Series')
axes[1].bar(range(len(vclass)), vclass.values, color=COLORS['orange']); axes[1].set_xticks(range(len(vclass))); axes[1].set_xticklabels(vclass.index, rotation=45, ha='right'); axes[1].set_title('Vehicle Class (series)')
axes[2].bar(energy.index, energy.values, color=COLORS['green']); axes[2].set_title('Energy Type (series)')
for a in axes: a.set_ylabel('Series')
fig.tight_layout(); fig.savefig(os.path.join(FIG, 'stage2_distributions.png'), dpi=150, bbox_inches='tight'); plt.show()
print('Energy share:'); print((energy/energy.sum()*100).round(1).to_string())


### 4.1 时间切分与评估子集

In [ ]:
print('In-population (Leg B) series :', manifest['n_series'])
print('Leg A series (annual sales)  :', int(feat[(feat.annual_sales.notna()) & (feat.year.between(2022, 2026))]['series_name'].nunique()))
print('Excluded sales-only series   :', len(set(sales.series_name.unique()) - set(feat.series_name.unique())))
print('\nStage 2 complete: EDA on the new 371-series in-population panel + 766-series config table.')


### 4.2 多模型对比（test 2026-01~2026-06，6 个月）

指标：WMAPE（体积加权 + per-series 中位数双口径）、MAPE、RMSE、MAE。体积加权 WMAPE 按实际销量加权，避免少量爆款主导。

### 4.3 特征消融（XGBoost）

对比「完整（lag+config）」「去掉滞后」「去掉配置」三版，量化历史销量滞后特征与配置特征的贡献。

In [ ]:
# Stage 3 uses absolute-time splits (no leakage). Evaluation subset = 150 series.
print('Train window :', manifest['time_cutoffs']['train_end'], '(usable rows', manifest['n_rows']['train_usable'], ')')
print('Val  window  :', manifest['time_cutoffs']['val_end'])
print('Test window  :', manifest['time_cutoffs']['test_end'], '(', manifest['n_rows']['test'], 'rows)')
print('Eval subset  : 150 series (stratified by energy x class x volume quartile)')


### 4.4 阶段三产出汇总

In [ ]:
# Load the real Stage-3 model comparison (6-month temporal test, volume-weighted WMAPE).
cmp = pd.read_csv(os.path.join(STAGE3, 'model_comparison.csv'))
cmp = cmp.sort_values('WMAPE_vol')
print(cmp.to_string(index=False))
print('\nBest model (volume-weighted WMAPE):', cmp.iloc[0]['model'], '%.1f%%' % cmp.iloc[0]['WMAPE_vol'])

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(cmp['model'], cmp['WMAPE_vol'], color=[COLORS['green'] if m=='XGBoost' else COLORS['blue'] for m in cmp['model']])
ax.set_title('Stage 3 - Model Comparison (volume-weighted WMAPE, test 2026-01~06)')
ax.set_xlabel('WMAPE_vol (%)'); ax.set_ylabel('Model')
for b, v in zip(bars, cmp['WMAPE_vol']):
    ax.annotate('%.1f%%' % v, xy=(b.get_width(), b.get_y()+b.get_height()/2),
                xytext=(4, 0), textcoords='offset points', va='center', fontsize=10)
fig.tight_layout(); fig.savefig(os.path.join(FIG, 'model_comparison.png'), dpi=150, bbox_inches='tight'); plt.show()


## 5. 阶段四：配置→销量归因（腿A）

阶段四回答「什么样的配置卖得好、配置能在多大程度解释销量差异」。在车系×年横截面上用 XGBoost 归因，`GroupKFold(5) by series_name` 防泄露。

In [ ]:
# XGBoost feature ablation: lag vs config contribution (volume-weighted WMAPE per version).
ab = pd.read_csv(os.path.join(STAGE3, 'xgb_ablation.csv'))
ab = ab.dropna(subset=['WMAPE'])
summ = ab.groupby('version')['WMAPE'].agg(['median','mean','count'])
print('XGBoost ablation (per-series WMAPE by version):')
print(summ.to_string())
print('\nDocumented volume-weighted WMAPE: FULL 63.2% | NO-LAG 73.2% | NO-CONFIG 48.8%')
print('-> Historical lag features dominate; config adds little/no positive contribution (leakage fixed).')


### 5.1 方法与数据

In [ ]:
print('Leg B baseline = XGBoost, volume-weighted WMAPE 63.2% (per-series median 59.4%).')
print('Fusion (Prophet 0.297 + XGBoost 0.703) = 62.2%; ARIMA 91.7%; LSTM 85.1%.')
print('Stage 3 complete: monthly forecasting baseline established (no sentiment yet).')


### 5.2 归因结果：R² 递进 + 特征重要性

R² 由仅年 **0.089** → +品牌 **0.154** → +配置 **0.303**（配置增量 ΔR² = +0.149）；特征重要性配置 **76.1%** / 品牌 **22.5%** / 年 **1.4%**。

### 5.3 阶段四产出汇总

In [ ]:
# Leg A: config -> annual sales cross-section. y = log1p(annual_sales).
# GroupKFold(5) by series_name (config near-constant within a series across years).
legA_df = feat[(feat.annual_sales.notna()) & (feat.year.between(2022, 2026))].copy()
print('Leg A rows :', len(legA_df), '| series:', legA_df.series_name.nunique())
y = np.log1p(legA_df['annual_sales'])
print('y = log1p(annual_sales): mean %.3f, std %.3f' % (y.mean(), y.std()))


## 6. 阶段五：舆情融合预测与话题预警（Phase B）

阶段五把舆情动态加入销量预测模型，检验能否提升精度、哪些话题需要预警。**当前为 Phase B 待接入新数据**：旧管线（669 系）经验显示动态情感未提升 volume-weighted 精度，但对尾部小销量车系可降低 per-series WMAPE。

In [ ]:
# R2 progression and feature importance from the real attribution run.
abl = pd.read_csv(os.path.join(STAGE4, 'config_attribution_ablation.csv'))
print('R2 progression (GroupKFold log-scale):')
for _, r in abl.iterrows():
    print('  %-11s R2=%.3f  (n_feat=%d)' % (r['variant'], r['R2_log_mean'], int(r['n_features'])))
print('\nConfig increment DeltaR2 = +%.3f (YEAR-ONLY -> +CONFIG)' % (
    abl.set_index('variant').loc['+CONFIG','R2_log_mean'] - abl.set_index('variant').loc['YEAR-ONLY','R2_log_mean']))

imp = pd.read_csv(os.path.join(STAGE4, 'config_importance.csv'))
imp.columns = ['feature','gain']
top = imp.sort_values('gain', ascending=False).head(15)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top['feature'][::-1], top['gain'][::-1], color=COLORS['teal'])
ax.set_title('Leg A - Top Feature Importance (config -> sales)')
ax.set_xlabel('Gain')
fig.tight_layout(); fig.savefig(os.path.join(FIG, 'stage4_shap_summary.png'), dpi=150, bbox_inches='tight'); plt.show()
print('\nDocumented grouped importance: config 76.1% / brand 22.5% / year 1.4%.')


### 6.1 旧管线经验（669 系）

In [ ]:
print('Leg A: config explains BETWEEN-SERIES sales differences (R2 up to 0.303).')
print('Same-series year-over-year moves are driven by non-config factors (generation, offers, WOM).')
print('Stage 4 complete: config->sales attribution established.')


### 6.2 新口径计划

### 6.3 阶段五产出汇总

In [ ]:
# Phase B: sentiment fusion forecasting (pending new-data integration).
# Legacy experience (669-series old pipeline): XGBoost-baseline 34.79% vs +Top3sent 35.21% volume-weighted.
print('Phase B plan: re-align sentiment to the new 371/736 series by series_name,')
print('then add dynamic sentiment as an exogenous variable to Leg B / Leg A.')
print('Legacy (669-series) volume-weighted: baseline 34.79% vs +Top3sent 35.21% (no lift).')
print('Tail benefit: per-series WMAPE 327% -> 311% for small-volume series.')


## 7. 阶段六：交互式网页看板

阶段六用 HTML + ECharts 搭建 7 屏纯静态交互式看板（项目概览、销量预测、舆情 ABSA、销量归因、舆情↔销量关系、舆情预警、品牌/车型钻取），数据由 `app/build_dashboard_data.py` 预烘焙为 JSON。

In [ ]:
print('Stage 5 (sentiment fusion) = Phase B, pending new-data integration.')
print('Methodology ready; will run on the new 371/736 series scope after sentiment re-alignment.')


### 7.1 看板结构

In [ ]:
# Stage 6 dashboard: pure static HTML + ECharts, data pre-baked to JSON.
print('Dashboard pages (app/): overview, forecast, absa, attribution, relation, alerts, drilldown.')
print('Data bridge  : app/static/data/*.json (from data/processed_new/* + data/sentiment/*).')
print('Launch       : cd app && python -m http.server 8000')
print('NOTE: current data bridge is an initial snapshot; Phase B will refresh to 371/736 scope.')


## 8. 结论与后续工作

**已完成（无舆情基线）**：

- 阶段一：数据准备（太平洋月度销量 + 汽车之家/太平洋配置，按 `series_name` 对齐）
- 阶段二：筛选与探索性可视化（371 in-population / 736 年度归因子集）
- 阶段三：腿B 月度预测（XGBoost 体积加权 WMAPE **63.2%**）
- 阶段四：腿A 配置归因（R² **0.303**，配置增量 +0.149）

**后续（Phase B）**：把 490 系懂车帝舆情按 `series_name` 重新对齐到 371/736 新口径，接入腿B/腿A 框架，刷新看板数据桥。

In [ ]:
print('=== China Auto Market Analysis - no-sentiment baseline (done) ===')
print('Leg B monthly forecasting : XGBoost 63.2% vol-WMAPE (371 in-population series).')
print('Leg A config attribution   : R2 0.303 (config 76.1% / brand 22.5% / year 1.4%).')
print('Alignment                  : 1017 sales x 766 config -> 371 Leg B + 736 Leg A; 646 excluded.')
print('Sentiment (Phase B)        : 490-series dongchedi collection ready, pending re-alignment.')
